### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

    1. Tracking agent behavior with logging, analytics, and debugging.
    2. Transforming prompts, tool selection, and output formatting.
    3. Adding retries, fallbacks and early termination logic.
    4. Applying rate limit, guardrails and PII detection.

In [1]:
from langchain_ollama import ChatOllama

# ollama model
ollama_model_agent = ChatOllama(model="qwen3:4b", temperature=0)
ollama_model_agent

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, model='qwen3:4b', temperature=0.0)

### Summarization MiddelWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following.

    1. Long-running conversations that execeed context windows.
    2. Multi-turn dialogues with extensive history
    3. Applications where preserving full conversation context matters

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model=ollama_model_agent,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=ollama_model_agent,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [4]:
### Run with thread id
config = {"configurable": {"thread_id":"test-1"}}

In [5]:
# Alternative test data
questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4*4"
]

for q in questions:
    response = agent.invoke( { "messages": [HumanMessage(content = q)]
        },
        config
    )
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='d63310b2-8aa9-41ae-ae4b-42fc3719696a'), AIMessage(content="The answer to **2 + 2** is **4**.  \n\nThis is a fundamental result in basic arithmetic (addition of whole numbers). In all standard mathematical contexts (e.g., elementary math, algebra, programming), **2 + 2 = 4**.  \n\n### Why?\n- **2** represents two units.\n- Adding **two more units** (2) gives a total of **four units** (4).  \n\nThis is consistent across all real-world applications, educational systems, and computational systems (e.g., Python, Excel, etc.).  \n\nIf you're asking in a *different context* (e.g., binary, modular arithmetic, or a specific programming language), the answer might vary—but for standard arithmetic, **4** is always correct. 😊  \n\nLet me know if you'd like a deeper explanation!", additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-28T08:35:44.8121877Z', 'don

### Token Size

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 start, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, bussiness center
    3. Budget Stay - 3 star, $75/night, free"""

agent = create_agent(
    model = ollama_model_agent,
    tools = [search_hotel],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = ollama_model_agent,
            trigger = ("tokens",550),
            keep = ("tokens",200)
        )
    ]
)

### Run with thread id
config = {"configurable": {"thread_id":"test-1"}}

# Token counter(approximate)
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token 

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~119 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='38ec3498-5627-4c7c-a894-664b6ed52fa1'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-28T08:59:29.7336007Z', 'done': True, 'done_reason': 'stop', 'total_duration': 23567419400, 'load_duration': 6400511300, 'prompt_eval_count': 140, 'prompt_eval_duration': 3141255000, 'eval_count': 123, 'eval_duration': 14006214000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a0e73d-4031-7111-86ed-02d358921681-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': '46e93299-5d04-4c14-8711-89b6aa142c73', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 140, 'output_tokens': 123, 'total_tokens': 263}), ToolMessage(content='Hotels in Paris:\n    1. Grand Hotel - 5 start, $350/night, spa, pool, gym\n    2. City Inn - 4 star, $180/night

### Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 start, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, bussiness center
    3. Budget Stay - 3 star, $75/night, free"""

agent = create_agent(
    model = ollama_model_agent,
    tools = [search_hotel],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = ollama_model_agent,
            trigger = ("fraction", 0.005),# 0.5% = ~640 tokens
            keep = ("fraction", 0.002) # 0.2% = ~256 tokens
        )
    ]
)

### Run with thread id
config = {"configurable": {"thread_id":"test-1"}}

# Token counter(approximate)
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token 

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")